PCA 和 ICA 都是把 EEG 的多通道数据重新表示成一组“成分”，但它们追求的目标不同。

一句话先说清楚：

**PCA 追求“方差最大、彼此正交、彼此不相关”；ICA 追求“统计独立、尽量非高斯、尽量像真实源信号”。**

---

# 1. EEG 数据矩阵先怎么理解？

假设你的 EEG 有 64 个通道，记录了 100000 个时间点。

可以把数据写成矩阵：

$$
X \in \mathbb{R}^{64 \times 100000}
$$

其中：

```text
每一行：一个 EEG 通道
每一列：某一个时刻所有通道的电位值
```

EEG 记录到的不是纯净脑源，而是很多源混在一起：

```text
EEG 通道信号
= 脑活动源 + 眨眼源 + 肌电源 + 心电源 + 噪声源 的线性混合
```

ICA 的经典模型是：

$$
X = AS
$$

其中：

```text
X：观测到的 EEG 通道数据
S：真实潜在源信号，例如脑源、眼动源、肌电源
A：混合矩阵
```

ICA 想做的是找到一个解混矩阵：

$$
S \approx WX
$$

也就是从混合信号里分离出相对独立的成分。

---

# 2. PCA 是什么？

PCA，全称是 **Principal Component Analysis，主成分分析**。

它的目标是：

> 找到一组新的坐标轴，让数据投影到这些轴上以后，方差从大到小排列。

也就是说，PCA 关心的是：

```text
哪个方向上的数据变化最大？
哪个方向携带最多方差信息？
```

比如二维数据是一团椭圆云：

```text
原始坐标轴：x 轴、y 轴
PCA 新坐标轴：沿着椭圆最长方向、次长方向
```

第一主成分 PC1 是方差最大的方向。

第二主成分 PC2 是在和 PC1 正交的条件下，方差第二大的方向。

---

## PCA 的数学核心

PCA 一般会对数据协方差矩阵做特征值分解。

假设数据已经去均值：

$$
C = \frac{1}{N-1}XX^T
$$

然后分解：

$$
C = U\Lambda U^T
$$

其中：

```text
C：协方差矩阵
U：特征向量矩阵，也就是 PCA 的新坐标轴
Λ：特征值矩阵，表示每个方向上的方差大小
```

PCA 变换是：

$$
Z = U^T X
$$

其中：

```text
Z：PCA 后的主成分信号
```

---

# 3. ICA 是什么？

ICA，全称是 **Independent Component Analysis，独立成分分析**。

它的目标是：

> 找到一组新的方向，使得投影出来的成分之间尽可能统计独立。

ICA 不是找“方差最大”的方向，而是找：

```text
哪个方向上的信号最不像高斯分布？
哪个方向最可能对应一个独立源？
```

为什么是“最非高斯”？

因为中心极限定理告诉我们：

> 多个独立源混合之后，混合信号会比单个源更接近高斯分布。

所以 ICA 的思路是：

```text
混合信号通常更高斯
真实独立源通常更非高斯
所以要找让投影结果最非高斯的方向
```

在 EEG 中，典型 ICA 成分可能是：

```text
眨眼成分
水平眼动成分
心电成分
肌电成分
某些脑源成分
线噪声成分
```

---

# 4. PCA 和 ICA 的根本区别

## PCA 追求不相关

PCA 得到的各个主成分之间满足：

$$
\operatorname{Cov}(PC_i, PC_j) = 0, \quad i \ne j
$$

也就是：

```text
PC1 和 PC2 的线性相关性为 0
```

但“不相关”只是弱条件。

两个信号不相关，不代表它们独立。

---

## ICA 追求独立

ICA 追求的是：

$$
p(s_1, s_2, ..., s_n) = p(s_1)p(s_2)...p(s_n)
$$

也就是联合概率可以拆成各自概率的乘积。

这比“不相关”强得多。

```text
不相关：只说明线性关系为 0
独立：说明几乎所有统计关系都没有
```

所以：

```text
独立 ⇒ 不相关
不相关 ⇏ 独立
```

---

# 5. 用一个例子理解 PCA 和 ICA

假设有两个真实源信号：

```text
s1：眨眼信号
s2：脑电 alpha 节律
```

它们通过头皮、电极、体积传导混在两个 EEG 通道里：

```text
x1 = 0.8s1 + 0.3s2
x2 = 0.4s1 + 0.9s2
```

PCA 会问：

```text
数据在哪个方向变化最大？
```

如果眨眼幅值特别大，PCA 的第一主成分很可能主要是眨眼。

但是 PCA 不保证把眨眼和脑电完全分开。

ICA 会问：

```text
哪个方向投影出来最像一个独立源？
```

它可能会把：

```text
IC1 ≈ 眨眼源
IC2 ≈ alpha 脑电源
```

所以在 EEG 去伪迹中，ICA 通常比 PCA 更直接有用。

---

# 6. PCA 和 ICA 的关系：PCA 常常是 ICA 的前置步骤

在实际算法里，PCA 经常先于 ICA 出现。

典型流程是：

```text
原始 EEG 数据 X
  ↓
去均值
  ↓
PCA
  ↓
白化 whitening
  ↓
ICA
  ↓
得到独立成分 ICs
```

原因是：

**PCA 可以先把数据变成不相关、单位方差的形式，从而简化 ICA 的搜索空间。**

---

# 7. 什么是白化？为什么 PCA 能帮助 ICA？

白化的目标是让数据满足：

$$
\operatorname{Cov}(Z) = I
$$

意思是：

```text
每个方向方差都变成 1
不同方向之间协方差都变成 0
```

也就是数据云从椭圆变成近似球形。

PCA 白化通常写作：

$$
Z = \Lambda^{-1/2} U^T X
$$

其中：

```text
U^T X：先旋转到 PCA 坐标系
Λ^{-1/2}：再把每个方向按方差缩放成单位方差
```

白化之后，ICA 不再需要搜索任意线性变换矩阵，而主要只需要搜索旋转矩阵。

这就是你之前问过的：

```text
白化前：要找一般矩阵
白化后：主要找旋转方向
```

---

# 8. 为什么白化后 ICA 主要只需要旋转？

白化前，数据可能是一个椭圆云。

要把混合信号分离出来，算法可能需要做：

```text
拉伸
压缩
剪切
旋转
```

这些都包含在一般矩阵里。

白化后，数据已经被 PCA 处理成球形：

```text
各方向方差相同
各方向不相关
```

这时如果再随便拉伸或压缩，就会破坏白化结果。

所以剩下的主要自由度就是：

```text
旋转这个球
```

二维情况下，旋转只需要一个角度。

三维情况下，旋转矩阵有 3 个自由度。

n 维情况下，正交旋转矩阵有：

$$
\frac{n(n-1)}{2}
$$

个自由度。

所以严格来说：

```text
二维白化后：只需要找 1 个旋转角
三维白化后：需要找 3 个旋转自由度
n 维白化后：需要找 n(n-1)/2 个旋转自由度
```

不是所有情况下都只有一个角度，只有二维例子里是这样。

---

# 9. PCA 降维和 ICA 的关系

在 MNE 里做 ICA 时，经常会看到类似概念：

```python
ica = ICA(n_components=20)
```

如果你有 64 个 EEG 通道，但设置：

```python
n_components=20
```

这通常意味着：

```text
先用 PCA 把数据从 64 维压到 20 维
然后在这 20 维空间里做 ICA
```

所以 ICA 里面常常内置 PCA。

这时 PCA 的作用有两个：

```text
1. 白化数据，方便 ICA 求解
2. 降维，减少噪声和计算量
```

但降维要谨慎。

如果你把维度降得太低，可能会丢掉一些有用成分或伪迹成分。

---

# 10. PCA 成分和 ICA 成分的含义不同

这是非常重要的一点。

## PCA 成分

PCA 成分按照方差大小排序：

```text
PC1：方差最大
PC2：方差第二大
PC3：方差第三大
...
```

所以 PCA 的排序有明确含义。

但是：

```text
PC1 不一定是最重要的脑源
PC1 可能只是最大伪迹，例如眨眼
```

---

## ICA 成分

ICA 成分通常没有天然的“方差大小排序”。

虽然软件可能会给 IC0、IC1、IC2 编号，但这个编号更多是算法输出顺序，不一定代表：

```text
IC0 最重要
IC1 第二重要
IC2 第三重要
```

在 MNE 中，ICA 成分常常和 PCA 空间、解释方差等有关，但你不能简单认为：

```text
ICA001 一定比 ICA004 更重要
```

判断 ICA 成分要看：

```text
成分地形图 topography
时间序列
频谱
与 EOG/ECG 通道的相关性
成分激活是否像眨眼/心跳/肌电
```

---

# 11. EEG 中 PCA 和 ICA 分别用来干什么？

## PCA 在 EEG 中常见用途

PCA 可以用于：

```text
降维
去噪
压缩数据
作为 ICA 的预处理步骤
构造白化数据
提取主要变化模式
```

但是 PCA 不太适合直接做眼动伪迹去除，因为它不保证分离出独立生理源。

---

## ICA 在 EEG 中常见用途

ICA 更常用于：

```text
去眨眼伪迹
去水平眼动伪迹
去心电伪迹
去肌电伪迹
分离相对独立的脑源活动
```

在 MNE 中，典型流程是：

```python
raw_for_ica = raw.copy()
raw_for_ica.filter(l_freq=1.0, h_freq=None)
raw_for_ica.set_eeg_reference("average")

ica = ICA(
    n_components=20,
    method="fastica",
    random_state=42
)

ica.fit(raw_for_ica, picks="eeg")
```

这里虽然你写的是 ICA，但内部通常会涉及 PCA/白化步骤。

---

# 12. PCA 和 ICA 的输出矩阵有何区别？

## PCA 输出

PCA 输出的是主成分：

$$
Z = U^T X
$$

这里的每一行是一个 PC。

它们满足：

```text
彼此正交
彼此不相关
按照方差大小排序
```

---

## ICA 输出

ICA 输出的是独立成分：

$$
S = WX
$$

这里的每一行是一个 IC。

它们尽量满足：

```text
彼此统计独立
分布尽量非高斯
不要求正交
不一定按方差排序
```

---

# 13. PCA 和 ICA 在“方向”上的区别

你之前问过 ICA 的“方向”是什么意思，这里可以和 PCA 对比起来看。

## PCA 的方向

PCA 找的是：

```text
数据云最拉长的方向
```

也就是方差最大的方向。

例如：

```text
一团椭圆形数据
PCA 第一方向 = 椭圆长轴
PCA 第二方向 = 椭圆短轴
```

---

## ICA 的方向

ICA 找的是：

```text
投影后最非高斯、最像独立源的方向
```

这个方向不一定是方差最大的方向。

例如眨眼幅度很大时：

```text
PCA 可能很容易把它放在 PC1
ICA 也可能找到它
```

但如果某个伪迹幅度不大，只是统计形状很特殊：

```text
PCA 未必重视它
ICA 可能仍然能分出来
```

因为 ICA 看的是独立性和非高斯性，而不是只看方差大小。

---

# 14. 为什么 ICA 之前不能完全靠 PCA 去伪迹？

因为 PCA 的成分只是“最大方差方向”。

假设 EEG 中有：

```text
强 alpha 节律
中等眨眼
微弱肌电
背景噪声
```

PCA 可能按方差排序：

```text
PC1：alpha 或眨眼
PC2：另一个大方差信号
PC3：混合的脑电/伪迹
...
```

它不保证：

```text
某一个 PC = 纯眨眼
某一个 PC = 纯脑电
```

PCA 的每个成分仍然可能是多个源的混合。

ICA 则更希望得到：

```text
IC1：眨眼
IC2：水平眼动
IC3：心跳
IC4：alpha 脑节律
...
```

当然 ICA 也不是完美的，但比 PCA 更符合“源分离”的目标。

---

# 15. 在 MNE 里怎么理解 `n_components`？

比如：

```python
ica = ICA(n_components=20, method="fastica")
ica.fit(raw_for_ica, picks="eeg")
```

如果你有 59 个 EEG 通道，那么：

```text
原始 EEG 空间：59 维
ICA 设置：20 个成分
```

大致可以理解为：

```text
先用 PCA 把 59 维数据压缩/白化到 20 维
再在这 20 维空间中做 ICA
最后得到 20 个 IC
```

如果写：

```python
ica = ICA(n_components=None)
```

则通常会尽量保留可用维度，但具体数量还会受数据秩 rank、投影器、坏道、平均参考等因素影响。

---

# 16. PCA 和 ICA 的关系总结表

| 对比项           | PCA         | ICA          |
| ------------- | ----------- | ------------ |
| 中文名           | 主成分分析       | 独立成分分析       |
| 目标            | 最大化方差       | 最大化独立性/非高斯性  |
| 成分关系          | 不相关、正交      | 尽量统计独立       |
| 是否按方差排序       | 是           | 通常不是         |
| 是否适合源分离       | 一般          | 更适合          |
| 是否常用于 EEG 去伪迹 | 辅助作用        | 核心方法         |
| 是否用于 ICA 内部   | 是，经常用于白化/降维 | 本身是后续分离步骤    |
| 对高斯信号         | 可以处理        | 难以分离多个高斯源    |
| 典型用途          | 降维、白化、去噪    | 眼动/心电/肌电伪迹分离 |

---

# 17. 最关键的理解

你可以这样记：

```text
PCA：先把混乱的数据坐标系摆正，让它变成不相关、按方差排序的坐标。
ICA：再在这个空间里旋转，寻找最像独立源的方向。
```

更形象一点：

```text
PCA 负责“整理空间”
ICA 负责“分离源信号”
```

在 EEG ICA 里，PCA 往往不是最终目的，而是 ICA 的准备步骤。

最终你在 `ica.plot_components()` 里看到的那些：

```text
ICA000
ICA001
ICA002
...
```

它们不是 PCA 主成分，而是经过 PCA 白化之后，再由 ICA 分离出来的独立成分。
